In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from catboost import CatBoostClassifier

In [6]:
# ==============================
# 1. 데이터 읽기
# ==============================
df = pd.read_csv("IBM_Telco-Customer-Churn.csv")

# 데이터 확인
print("데이터 크기:", df.shape)
print(df.head())


데이터 크기: (7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies  

In [7]:
# ==============================
# 2. 전처리
# ==============================

# 2-1. customerID 제거
df = df.drop(columns=["customerID"])

# 2-2. TotalCharges 숫자형 변환
# 공백 등 숫자로 변환 안 되는 값은 NaN 처리
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# 2-3. 결측치 확인
print("\n[결측치 개수]")
print(df.isnull().sum())

# 2-4. 결측치 처리
# TotalCharges 결측치는 중앙값으로 대체
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# 2-5. 타겟 변수 변환
# Churn: Yes -> 1, No -> 0
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})



[결측치 개수]
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64


In [8]:
# ==============================
# 3. X, y 분리
# ==============================
X = df.drop(columns=["Churn"])
y = df["Churn"]

# 범주형 컬럼 찾기
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("\n[범주형 컬럼]")
print(cat_cols)

# CatBoost에 넣을 범주형 컬럼 인덱스
cat_features = [X.columns.get_loc(col) for col in cat_cols]

print("\n[범주형 컬럼 인덱스]")
print(cat_features)


[범주형 컬럼]
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

[범주형 컬럼 인덱스]
[0, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]


C:\Users\Playdata\AppData\Local\Temp\ipykernel_17564\1934805101.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns.tolist()


In [9]:
# ==============================
# 4. train / test 분리
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\n[학습 데이터 타겟 비율]")
print(y_train.value_counts(normalize=True))

print("\n[테스트 데이터 타겟 비율]")
print(y_test.value_counts(normalize=True))



X_train shape: (5634, 19)
X_test shape: (1409, 19)

[학습 데이터 타겟 비율]
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64

[테스트 데이터 타겟 비율]
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


In [10]:

# ==============================
# 5. CatBoost 모델 생성
# ==============================
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100
)

In [11]:
# ==============================
# 6. 모델 학습
# ==============================
model.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=(X_test, y_test),
    use_best_model=True
)

0:	test: 0.7948113	best: 0.7948113 (0)	total: 195ms	remaining: 1m 37s
100:	test: 0.8450606	best: 0.8457646 (59)	total: 4.11s	remaining: 16.2s
200:	test: 0.8452673	best: 0.8457646 (59)	total: 7.88s	remaining: 11.7s
300:	test: 0.8456147	best: 0.8459725 (216)	total: 13.5s	remaining: 8.93s
400:	test: 0.8441706	best: 0.8459725 (216)	total: 20.3s	remaining: 5s
499:	test: 0.8413741	best: 0.8459725 (216)	total: 26.7s	remaining: 0us

bestTest = 0.8459725129
bestIteration = 216

Shrink model to first 217 iterations.


In [14]:
# ==============================
# 7. 예측
# ==============================
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

In [15]:
# ==============================
# 8. 성능 평가
# ==============================
acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("\n==============================")
print("모델 성능 평가")
print("==============================")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\n[Classification Report]")
print(classification_report(y_test, y_pred))

print("\n[Confusion Matrix]")
print(confusion_matrix(y_test, y_pred))



모델 성능 평가
Accuracy : 0.8070
Precision: 0.6711
Recall   : 0.5348
F1-score : 0.5952
ROC-AUC  : 0.8460

[Classification Report]
              precision    recall  f1-score   support

           0       0.84      0.91      0.87      1035
           1       0.67      0.53      0.60       374

    accuracy                           0.81      1409
   macro avg       0.76      0.72      0.73      1409
weighted avg       0.80      0.81      0.80      1409


[Confusion Matrix]
[[937  98]
 [174 200]]


In [16]:
# ==============================
# 9. Feature Importance 확인
# ==============================
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.get_feature_importance()
}).sort_values(by="Importance", ascending=False)

print("\n[Feature Importance Top 20]")
print(feature_importance.head(20))


[Feature Importance Top 20]
             Feature  Importance
14          Contract   18.057764
4             tenure   15.729257
7    InternetService   12.632343
17    MonthlyCharges    7.493492
18      TotalCharges    7.405038
16     PaymentMethod    5.926039
6      MultipleLines    4.972698
8     OnlineSecurity    4.744769
11       TechSupport    4.695068
9       OnlineBackup    4.291974
12       StreamingTV    2.606315
15  PaperlessBilling    2.603532
13   StreamingMovies    2.386612
3         Dependents    1.824432
0             gender    1.476754
10  DeviceProtection    1.415956
2            Partner    0.698091
1      SeniorCitizen    0.689689
5       PhoneService    0.350177
